# Phase 0 — Chargement C-MAPSS et confirmation de la structure

Objectif : charger FD001 et FD002 (train/test/RUL) dans des DataFrames pandas, et vérifier
que les tailles et la structure des colonnes correspondent au `readme.txt` fourni avec le jeu de données.

Critère « fait » (plan d'exécution, Phase 0) : fichiers lus dans un DataFrame + tableau des sous-ensembles confirmé.

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../DATA/CMaps")

# Le readme.txt annonce "26 columns" et énumère les colonnes jusqu'à "26) sensor
# measurement 26", mais 2 (unit, cycle) + 3 (réglages) + 26 (capteurs) ferait 31
# colonnes, pas 26. Coquille connue du readme original PHM08 : en comptant les
# colonnes réelles (6 à 26 inclus), il y a 21 capteurs pour un total de 26 colonnes.
INDEX_NAMES = ["unit_number", "time_cycles"]
SETTING_NAMES = ["setting_1", "setting_2", "setting_3"]
SENSOR_NAMES = [f"sensor_{i}" for i in range(1, 22)]
COL_NAMES = INDEX_NAMES + SETTING_NAMES + SENSOR_NAMES

In [2]:
def load_split(subset: str, split: str) -> tuple[pd.DataFrame, int]:
    """Charge train_{subset}.txt ou test_{subset}.txt et nomme les colonnes.

    Retourne aussi le nombre de colonnes trouvées dans le fichier brut,
    pour pouvoir vérifier qu'il correspond bien aux 26 attendues.
    """
    path = DATA_DIR / f"{split}_{subset}.txt"
    df = pd.read_csv(path, sep=r"\s+", header=None)
    n_cols_found = df.shape[1]
    df = df.iloc[:, : len(COL_NAMES)]
    df.columns = COL_NAMES
    return df, n_cols_found


def load_rul(subset: str) -> pd.DataFrame:
    """Charge le vecteur RUL vrai du jeu de test (RUL_{subset}.txt)."""
    path = DATA_DIR / f"RUL_{subset}.txt"
    return pd.read_csv(path, sep=r"\s+", header=None, names=["RUL"])

In [3]:
# Valeurs attendues d'après le readme (nombre de trajectoires par sous-ensemble)
EXPECTED_UNITS = {
    "FD001": {"train": 100, "test": 100},
    "FD002": {"train": 260, "test": 259},
}

datasets = {}
rows = []

for subset, expected in EXPECTED_UNITS.items():
    train_df, train_ncols = load_split(subset, "train")
    test_df, test_ncols = load_split(subset, "test")
    rul_df = load_rul(subset)
    datasets[subset] = {"train": train_df, "test": test_df, "rul": rul_df}

    rows.append({
        "subset": subset,
        "train_units_found": train_df["unit_number"].nunique(),
        "train_units_expected": expected["train"],
        "test_units_found": test_df["unit_number"].nunique(),
        "test_units_expected": expected["test"],
        "rul_rows": len(rul_df),
        "train_cols_in_file": train_ncols,
        "test_cols_in_file": test_ncols,
    })

summary = pd.DataFrame(rows)
summary

,subset,train_units_found,train_units_expected,test_units_found,test_units_expected,rul_rows,train_cols_in_file,test_cols_in_file
0,FD001,100,100,100,100,100,26,26
1,FD002,260,260,259,259,259,26,26


In [4]:
# Confirmation explicite : unités trouvées = attendues, RUL alignée sur le nombre
# de moteurs de test, et 26 colonnes bien présentes dans chaque fichier brut.
for row in summary.itertuples():
    checks = {
        "train_units": row.train_units_found == row.train_units_expected,
        "test_units": row.test_units_found == row.test_units_expected,
        "rul_matches_test_units": row.rul_rows == row.test_units_expected,
        "cols_count_26": row.train_cols_in_file == 26 and row.test_cols_in_file == 26,
    }
    status = "OK" if all(checks.values()) else "MISMATCH"
    print(f"{row.subset}: {status} -> {checks}")

FD001: OK -> {'train_units': True, 'test_units': True, 'rul_matches_test_units': True, 'cols_count_26': True}
FD002: OK -> {'train_units': True, 'test_units': True, 'rul_matches_test_units': True, 'cols_count_26': True}
